In [1]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
import pandas as pd

/Users/matt/Documents/Algo-Trading/env/lib/python3.13/site-packages/pytorch_forecasting/models/base/_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
df = pd.read_csv('../data/preproc_coins.csv')
print(f'Total rows: {len(df):,}')
print(f'Tokens: {df["symbol"].nunique()}')
print(f'Time range: {df["time_idx"].min()} to {df["time_idx"].max():,}')
df.head()

Total rows: 736,065
Tokens: 19
Time range: 0 to 52,029


,time_idx,symbol,open_log,high_log,low_log,close_log,volume_log,btc_close_log,eth_close_log,eth_btc_ratio,hour_sin,hour_cos,norm_day_sin,norm_day_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,next_close_log
0,0,ADA/USD,0.042488,-0.035411,0.017805,-0.017555,14.620035,-0.005870,-0.004027,0.063330,0.965926,2.588190e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.020670
1,1,ADA/USD,-0.016604,-0.051752,-0.047637,-0.020670,14.582821,0.005249,0.004362,0.063273,1.000000,6.123234e-17,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,0.001324
2,2,ADA/USD,-0.021549,-0.003644,0.027403,0.001324,14.010766,0.014037,0.012739,0.063191,0.965926,-2.588190e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.002611
3,3,ADA/USD,-0.001014,-0.011549,-0.010253,-0.002611,13.705804,-0.000842,0.004153,0.063508,0.866025,-5.000000e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.003713
4,4,ADA/USD,-0.000195,-0.000693,0.002810,-0.003713,13.441326,0.001574,0.003151,0.063608,0.707107,-7.071068e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,0.003986


# Train/Val/Test Split (Per Token)
Since each token has its own time series with different lengths, we split each token's data individually using the same percentage thresholds.

In [3]:
def split_by_token(group, train_pct=0.70, val_pct=0.15):
  """
  Split time series sequentially to prevent data leakage.
  Train: earliest 70% of data
  Val: next 15% of data
  Test: latest 15% of data

  This mimics production: training on past, predicting future.
  """
  max_idx = group['time_idx'].max()
  train_cutoff = int(train_pct * max_idx)
  val_cutoff = int((train_pct + val_pct) * max_idx)

  group['split'] = 'test'  # default
  group.loc[group['time_idx'] <= train_cutoff, 'split'] = 'train'
  group.loc[
    (group['time_idx'] > train_cutoff) & (group['time_idx'] <= val_cutoff), 'split'
  ] = 'val'

  return group


# Apply split to each token
df = df.groupby('symbol', group_keys=False).apply(split_by_token)

# Verify splits
print('Split percentages:')
print(df['split'].value_counts(normalize=True) * 100)

Split percentages:
split
train    69.999796
test     15.000577
val      14.999626
Name: proportion, dtype: float64


/var/folders/zf/pd74vy996kg6l6dpbrpqm1jr0000gn/T/ipykernel_26651/1576318309.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('symbol', group_keys=False).apply(split_by_token)


In [4]:
# Create separate dataframes for each split
train_df = df[df['split'] == 'train'].copy().drop(columns=['split'])
val_df = df[df['split'] == 'val'].copy().drop(columns=['split'])
test_df = df[df['split'] == 'test'].copy().drop(columns=['split'])

print(f'Train: {len(train_df):,} rows')
print(f'Val: {len(val_df):,} rows')
print(f'Test: {len(test_df):,} rows')

Train: 515,244 rows
Val: 110,407 rows
Test: 110,414 rows


# Create TimeSeriesDataSet with RobustGroupNormalizer

In [5]:
# Define parameters
max_prediction_length = 1  # Predict 1 step ahead (next_close_log)
min_encoder_length = 168  # 1 week of hourly data
max_encoder_length = 168

# Financial features that need RobustGroupNormalizer (log returns)
financial_features = [
  'open_log',
  'high_log',
  'low_log',
  'close_log',
  'volume_log',
  'btc_close_log',
  'eth_close_log',
  'eth_btc_ratio',
]

# Cyclical temporal features (already bounded, no normalization needed)
temporal_features = [
  'hour_sin',
  'hour_cos',
  'norm_day_sin',
  'norm_day_cos',
  'weekday_sin',
  'weekday_cos',
  'month_sin',
  'month_cos',
  'year',
]

# Create scalers dictionary for financial features
kwargs = {'upper': 0.90, 'lower': 0.10}

scalers = {
  # Price log features - use ROBUST scaling due to outliers (especially low_log)
  'open_log': GroupNormalizer(
    groups=['symbol'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  'high_log': GroupNormalizer(
    groups=['symbol'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  'low_log': GroupNormalizer(
    groups=['symbol'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  'close_log': GroupNormalizer(
    groups=['symbol'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  'volume_log': GroupNormalizer(
    groups=['symbol'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  # BTC/ETH log returns - smaller ranges, but robust is still safer for crypto
  'btc_close_log': GroupNormalizer(
    groups=[],  # normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  'eth_close_log': GroupNormalizer(
    groups=[],  # normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  # ETH/BTC ratio - bimodal! But it's a regime signal so preserve it
  'eth_btc_ratio': GroupNormalizer(
    groups=[],  # normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
}


print(f'Encoder length: {max_encoder_length} timesteps')
print(f'Prediction length: {max_prediction_length} timesteps')
print(f'Financial features with RobustGroupNormalizer: {len(financial_features)}')

Encoder length: 168 timesteps
Prediction length: 1 timesteps
Financial features with RobustGroupNormalizer: 8


In [6]:
training = TimeSeriesDataSet(
  train_df,
  time_idx='time_idx',
  target='next_close_log',
  group_ids=['symbol'],
  # Sequence lengths
  max_encoder_length=max_encoder_length,
  max_prediction_length=max_prediction_length,
  min_encoder_length=min_encoder_length,
  # Known features (available at prediction time)
  time_varying_known_reals=temporal_features,
  # Unknown features (only known historically, including all financial features)
  time_varying_unknown_reals=financial_features,
  # Categorical features
  static_categoricals=['symbol'],
  # Target normalization
  target_normalizer=GroupNormalizer(
    groups=['symbol'],  # we want to normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs=kwargs,
  ),
  # Apply RobustGroupNormalizer to all financial features
  scalers=scalers,
  # Additional parameters
  add_relative_time_idx=True,
  add_target_scales=True,
  add_encoder_length=False,
  allow_missing_timesteps=False,
  randomize_length=False,
)

print('Training dataset created successfully!')
print(f'Number of samples: {len(training)}')

Training dataset created successfully!
Number of samples: 512052


In [ ]:
validation = TimeSeriesDataSet.from_dataset(training, val_df, stop_randomization=True)

print('Validation dataset created successfully!')
print(f'Number of samples: {len(validation)}')

Validation dataset created successfully!
Number of samples: 107215


In [8]:
test = TimeSeriesDataSet.from_dataset(
  training, test_df, predict=False, stop_randomization=True
)

print('Test dataset created successfully!')
print(f'Number of samples: {len(test)}')

Test dataset created successfully!
Number of samples: 107222


In [10]:
# Save the datasets
training.save('../data/bin/training_temporaldataset.pkl')
validation.save('../data/bin/validation_temporaldataset.pkl')
test.save('../data/bin/test_temporaldataset.pkl')

"""
To load:

from pytorch_forecasting import TimeSeriesDataSet
training = TimeSeriesDataSet.load('../data/training_temporaldataset.pkl')
"""
print('Saved all datasets to data/bin folder!')

Saved all datasets to data/bin folder!
